# Tutorial 4

## Setup

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

### install dependencies. Colab already has a kernel and most packages, so install
### only what is missing there: upgrading Colab's pinned packages (ipykernel,
### google-auth, jupyter-client, ...) can break the runtime
if IN_COLAB or not os.path.exists("requirements.txt"):
    %pip install -q "pydantic-ai-slim[openai]>=2.57.0" "langfuse>=5.0.2" python-dotenv sentence-transformers scikit-learn
else:
    %pip install -q -r requirements.txt


In [ ]:
import os
import numpy as np
import time
import pandas as pd

from dotenv import load_dotenv

### load environment variables: from the .env file (VS Code / local),
### or from Colab secrets (key icon in the left sidebar) when on Colab
load_dotenv()

try:
    from google.colab import userdata
    for key in ["OPENROUTER_API_KEY", "LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY", "LANGFUSE_BASE_URL"]:
        if not os.getenv(key):
            try:
                os.environ[key] = userdata.get(key)
            except Exception:
                print(f"Colab secret {key} not found - add it in the Secrets panel")
except ImportError:
    pass  ### not running on Colab

if not os.getenv("OPENROUTER_API_KEY"):
    print("OPENROUTER_API_KEY is not set - add it to .env (local) or Colab secrets")

In [ ]:
from pydantic_ai import Agent, RunContext
from pydantic_ai.exceptions import ModelHTTPError
from pydantic_ai.usage import UsageLimitExceeded, UsageLimits
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai.messages import ToolCallPart


### initialize Pydantic AI instrumentation
Agent.instrument_all()



## Langfuse

https://langfuse.com/integrations/gateways/openrouter

In [4]:
from langfuse import get_client
langfuse = get_client()

### verify connection
if langfuse.auth_check():
    print("Langfuse client is authenticated and ready!")
else:
    print("Authentication failed. Please check your credentials and host.")

Langfuse client is authenticated and ready!


## Baseline

In [ ]:
openrouter_provider = OpenAIProvider(
    base_url="https://openrouter.ai/api/v1",
    api_key = os.getenv("OPENROUTER_API_KEY")
)

### choose any model, can use a free one
### https://openrouter.ai/collections/free-models
openrouter_model = "anthropic/claude-haiku-5.5"


In [12]:
agent = Agent(
    model=OpenAIChatModel(model_name=openrouter_model, provider=openrouter_provider),
    instructions=(
        "You are a tutorial workshop assistant for LLM agents."
    )
)

query = ("Explain in 3 bullets: what is an LLM agent and how it differs from plain chat."
"Please think deeply to show your reasoning steps and summary.")

result = await agent.run(query)
print(result.output)

**Reasoning steps (internal chain‑of‑thought):**  
1. **Identify core components:**  
   - An LLM (Large Language Model) is a statistical predictor that generates text given a prompt.  
   - “Plain chat” refers to a straightforward interaction where the user sends a prompt and the model returns a single‑turn response, with no additional logic beyond the model’s innate generation.  
2. **Define what makes something an “agent”:**  
   - Agency implies the system can **act** toward goals, maintain **state** (memory), **plan** sequences of actions, and **use external tools** (APIs, code executors, databases) to affect the world.  
   - An LLM agent wraps the raw LLM in a control loop that decides *what* to ask the model, *when* to call tools, and *how* to update its internal state based on observations.  
3. **Contrast capabilities:**  
   - Plain chat: stateless (or only short‑term context window), no tool use, no explicit planning; the model’s output is limited to what can be inferred fr

Now check the Tracing section on Langfuse - you should see your traces there!

## Adding Tools

In [6]:
import ast
import json
import re
import typing

from dataclasses import dataclass
from datetime import datetime
from typing import Any
from zoneinfo import ZoneInfo

In [80]:
def safe_calculator(expression: str) -> dict[str, Any]:
    try:
        tree = ast.parse(str(expression), mode="eval")
        allowed = (
            ast.Expression,
            ast.BinOp,
            ast.UnaryOp,
            ast.Add,
            ast.Sub,
            ast.Mult,
            ast.Div,
            ast.Mod,
            ast.Pow,
            ast.USub,
            ast.UAdd,
            ast.Constant,
        )
        for node in ast.walk(tree):
            if not isinstance(node, allowed):
                raise ValueError(f"Disallowed syntax: {type(node).__name__}")
        value = eval(compile(tree, filename="<calc>", mode="eval"), {"__builtins__": {}}, {})
        return {"expression": expression, "value": value}
    except Exception as exc:
        return {"expression": expression, "error": str(exc)}

def celsius_to_fahrenheit(celsius: float) -> float:
    """Convert temperature from Celsius to Fahrenheit."""
    temp_fahrenheit = celsius * (9/5) + 32
    return temp_fahrenheit

def fahrenheit_to_celsius(fahrenheit: float) -> float:
    """Convert temperature from Fahrenheit to Celsius."""
    temp_celsius = (fahrenheit - 32) * (5/9)
    return temp_celsius

COURSE_DOCS = [
    {
        "title": "Section 1 - Neuroanatomy",
        "content": "What structures are in the brain, and how they are segmented.",
    },
    {
        "title": "Section 2 - Neuroimaging",
        "content": "How the brain can be measured, structural and functional scans.",
    },
    {
        "title": "Section 3 - Human Memory",
        "content": "How humans encode, store, and retrieve information.",
    },
]


def search_course_docs(query: str, top_k: int = 3) -> dict[str, Any]:
    tokens = set(re.findall(r"[a-zA-Z0-9]+", str(query).lower()))
    scored = []
    for doc in COURSE_DOCS:
        text = f"{doc['title']} {doc['content']}".lower()
        doc_tokens = set(re.findall(r"[a-zA-Z0-9]+", text))
        scored.append((len(tokens & doc_tokens), doc))
    scored.sort(key=lambda x: x[0], reverse=True)
    hits = [doc for score, doc in scored if score > 0][:top_k]
    return {"query": query, "hits": hits, "total_hits": len(hits)}

In [81]:
@dataclass
class LectureDeps:
    lecturer: str
    course: str

agent = Agent(
    model=OpenAIChatModel(model_name=openrouter_model, provider=openrouter_provider),
    instructions=(
        "You are a guest-lecture assistant."
        "Use tools for facts/computation. Keep answers concise and explicit."
    ),
    deps_type=LectureDeps
)

### pass this instance (not the LectureDeps class) as deps= on every run
lecture_deps = LectureDeps(lecturer="William James", course="Intro to Cognitive Neuroscience")

@agent.tool
def get_lecture_context(ctx: RunContext[LectureDeps]) -> dict[str, str]:
    """Return metadata about this lecture setup."""
    return {"lecturer": ctx.deps.lecturer, "course": ctx.deps.course}

@agent.tool_plain
def calculator(expression: str) -> dict[str, Any]:
    """Safely evaluate arithmetic expressions."""
    return safe_calculator(expression)

@agent.tool_plain
def search_course_docs_tool(query: str, top_k: int = 3) -> dict[str, Any]:
    """Search mini in-memory course notes."""
    return search_course_docs(query=query, top_k=top_k)

@agent.tool_plain
def unit_conversion(value: float, from_unit: str, to_unit: str) -> float:
    """Convert units between Celsius<->Fahrenheit."""
    from_unit = from_unit.lower()
    to_unit = to_unit.lower()

    if from_unit == "celsius" or from_unit == "c":
        return celsius_to_fahrenheit(value)
    elif from_unit == "fahrenheit" or from_unit == "f":
        return fahrenheit_to_celsius(value)
    else:
        raise ValueError(f"Unsupported unit conversion: {from_unit} to {to_unit}")

print(f"Agent ready with {len(agent._function_toolset.tools)} tools\n")
print("Tools registered on agent:")
for tool_name in agent._function_toolset.tools:
    print(f"  {tool_name}")

Agent ready with 4 tools

Tools registered on agent:
  get_lecture_context
  calculator
  search_course_docs_tool
  unit_conversion


In [13]:
query = """Use tools to do the following tasks:
(1) Which section in our course covers neuroimaging?
(2) What is 21 degrees celsius in fahrenheit?
(3) What is 34 * 33?
(4) Who is the lecturer of this course?"""

In [11]:
### tight limits on purpose: a run that exceeds them raises UsageLimitExceeded
try:
    result = await agent.run(
        query,
        deps=lecture_deps,
        usage_limits = UsageLimits(
            request_limit = 5,          ### max agent loop turns
            input_tokens_limit = 1000,  ### max input tokens
            output_tokens_limit = 300,  ### max output tokens
            total_tokens_limit = 1200,  ### max total tokens
            cost_limit = 0.05           ### max cost in USD
        )
        )

    print("---"*3)
    print(result.output)
except UsageLimitExceeded as e:
    print(f"Usage limit hit: {e}")


---------
(1) Section 2 – Neuroimaging  
(2) 21 °C = 69.8 °F  
(3) 34 × 33 = 1122  
(4) Lecturer: William James


## Within-Task Reliability

In [16]:
query = "Who is the lecturer of this course?"

In [17]:
outputs = []

for i in range(5):
    print(f"Running agent iteration {i+1}...")
    res = await agent.run(
        query,
        deps=lecture_deps,
        usage_limits = UsageLimits(
          request_limit = 5,
          cost_limit = 0.05
          )
        )
    outputs.append(res.output)
    print(f"Output {i+1}: {res.output}\n" + "-"*20)

Running agent iteration 1...
Output 1: The lecturer for this course is William James.
--------------------
Running agent iteration 2...
Output 2: The lecturer for this course is **William James**.
--------------------
Running agent iteration 3...
Output 3: The lecturer for this course is **William James**.
--------------------
Running agent iteration 4...
Output 4: The lecturer for this course is **William James**.
--------------------
Running agent iteration 5...
Output 5: The lecturer of this course is William James.
--------------------


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model_st = SentenceTransformer("all-MiniLM-L6-v2")

In [20]:
### compute embeddings for each of the outputs
embeddings = model_st.encode(outputs)

### calculate cosine similarity matrix
sim_matrix = cosine_similarity(embeddings)

print("\nPairwise Cosine Similarity Matrix (Sentence Transformers):")
print(sim_matrix)
print(f"\nAverage pairwise similarity: {sim_matrix[np.triu_indices(len(outputs), k=1)].mean():.4f}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Pairwise Cosine Similarity Matrix (Sentence Transformers):
[[1.0000004  0.94838107 0.94838107 0.94838107 0.99237925]
 [0.94838107 1.         1.         1.         0.95782495]
 [0.94838107 1.         1.         1.         0.95782495]
 [0.94838107 1.         1.         1.         0.95782495]
 [0.99237925 0.95782495 0.95782495 0.95782495 1.0000001 ]]

Average pairwise similarity: 0.9711


## Between-Task Reliability

In [32]:
queries = [
    "Which section in our course covers neuroimaging?",
    "Which section in our course covers functional scans?",
    "Which section in our course covers fMRI?",
    "Which section in our course covers the BOLD signal?"
]

expected_result = "Section 2"
normalized_expected = re.sub(r'\s+', ' ', expected_result)

In [36]:
outputs = []

for i in range(len(queries)):
    print(f"Running agent iteration {i+1}...")
    res = await agent.run(
        queries[i],
        deps=lecture_deps,
        usage_limits = UsageLimits(
          request_limit = 5,
          cost_limit = 0.05
          )
        )

    ### normalize non-standard whitespace characters to regular spaces
    normalized_output = re.sub(r'\s+', ' ', res.output)

    ### check for success (1) or failure (0)
    if normalized_expected in normalized_output:
        score = 1
    else:
        score = 0

    outputs.append((res.output, score))
    print(f"Output {i+1}: {res.output}")
    print(f"Contains Expected Result: {score}")
    print("-"*20)

Running agent iteration 1...
Output 1: Section 2 of the course covers neuroimaging (it discusses how the brain can be measured using structural and functional scans).
Contains Expected Result: 1
--------------------
Running agent iteration 2...
Output 2: Functional scans are covered in **Section 2 – Neuroimaging** of the course. The description for that section notes: “How the brain can be measured, structural and functional scans.”
Contains Expected Result: 1
--------------------
Running agent iteration 3...
Output 3: fMRI is covered in **Section 2 – Neuroimaging**, which discusses how the brain can be measured using structural and functional scans.
Contains Expected Result: 1
--------------------
Running agent iteration 4...
Output 4: I searched the searchable course materials for “BOLD signal,” “BOLD,” “fMRI,” and related terms, but no matching sections or notes were found. Based on the available lecture context (course: Intro to Cognitive Neuroscience; lecturer: William James), the

In [56]:
print(f"\n% Trials containing expected result: {np.mean([item[1] for item in outputs])*100}%")


% Trials containing expected result: 75.0%


In [51]:
### compute embeddings for each of the outputs
embeddings = model_st.encode([item[0] for item in outputs])

### calculate cosine similarity matrix
sim_matrix = cosine_similarity(embeddings)

print("\nPairwise Cosine Similarity Matrix (Sentence Transformers):")
print(sim_matrix)
print(f"\nAverage pairwise similarity: {sim_matrix[np.triu_indices(len(outputs), k=1)].mean():.4f}")


Pairwise Cosine Similarity Matrix (Sentence Transformers):
[[1.0000002  0.90199244 0.842262   0.5471158 ]
 [0.90199244 1.0000001  0.90663505 0.54279983]
 [0.842262   0.90663505 1.0000002  0.48501116]
 [0.5471158  0.54279983 0.48501116 1.        ]]

Average pairwise similarity: 0.7043


## Multi-Agent AI

In [82]:
teacher_agent = Agent(
    model=OpenAIChatModel(
        model_name=openrouter_model,
        provider=openrouter_provider
        ),
    instructions="You are the teacher who performs calculations."
    )

@teacher_agent.tool_plain
def calculator(expression: str) -> dict[str, Any]:
    """Safely evaluate arithmetic expressions."""
    return safe_calculator(expression)

course_coordinator_agent = Agent(
    model=OpenAIChatModel(
        model_name=openrouter_model,
        provider=openrouter_provider
        ),
    instructions="You are the course coordinator who knows the course content."
    )

@course_coordinator_agent.tool_plain
async def search_course_docs_tool(query: str, top_k: int = 3) -> dict[str, Any]:
    """Search mini in-memory course notes."""
    return search_course_docs(query=query, top_k=top_k)

In [83]:
planning_agent = Agent(
    model=OpenAIChatModel(
        model_name=openrouter_model,
        provider=openrouter_provider
        ),
    instructions=(
        "You are the planning agent. Break tasks into steps and choose the most suitable agent to execute the task."
        )
    )

TRACE: list[dict] = []  # cleared before each test case

def tool_names(result) -> list[str]:
    return [p.tool_name for m in result.all_messages()
            for p in m.parts if isinstance(p, ToolCallPart)]

@planning_agent.tool
async def teacher(ctx: RunContext[None], query: str) -> str:
    """Delegate calculation-related questions to the teacher agent."""
    res = await teacher_agent.run(query, usage=ctx.usage)
    TRACE.append({"to": "teacher", "query": query, "output": res.output,
                  "inner_tools": tool_names(res)})
    return res.output

@planning_agent.tool
async def docs_specialist(ctx: RunContext[None], query: str) -> str:
    """Delegate course questions to the course coordinator agent."""
    res = await course_coordinator_agent.run(query, usage=ctx.usage)
    TRACE.append({"to": "docs_specialist", "query": query, "output": res.output,
                  "inner_tools": tool_names(res)})
    return res.output

In [66]:
query = "What content is in the course section with the number equal to 12/4?"

In [67]:
### multi-agent run
result = await planning_agent.run(
    query
)
print("---"*3)
print(result.output)

---------
The course section with the number equal to \(12 \div 4 = 3\) is **Section 3**, which covers **Human Memory**, focusing on **how humans encode, store, and retrieve information**.


In [68]:
### single agent run
result = await agent.run(
    query,
    deps=lecture_deps
)
print("---"*3)
print(result.output)

---------
Section 3 of the course is titled **“Human Memory”** and its content covers **how humans encode, store, and retrieve information**.


## Coordination Metrics

In [72]:
CASES = [
    {"type": "calc",     "q": "What is 34 * 33?",
     "expect": {"teacher"}, "answer": ["1122"]},
    {"type": "docs",     "q": "Which section of the course covers neuroimaging?",
     "expect": {"docs_specialist"}, "answer": ["2"]},
    {"type": "handoff",  "q": "Which section number covers human memory? Raise that number to the 4th power.",
     "expect": {"docs_specialist", "teacher"}, "order": ["docs_specialist", "teacher"],
     "handoff": "3", "answer": ["81"]},
    {"type": "handoff",  "q": "How many sections does the course have? Multiply that by 7.",
     "expect": {"docs_specialist", "teacher"}, "order": ["docs_specialist", "teacher"],
     "handoff": "3", "answer": ["21"]},
    {"type": "parallel", "q": "Which section covers neuroanatomy, and what is 12 squared?",
     "expect": {"docs_specialist", "teacher"}, "answer": ["1", "144"]},
    {"type": "none",     "q": "Hi! In one sentence, what can you help me with?",
     "expect": set(), "answer": []},
]

In [ ]:
import asyncio

### free OpenRouter models allow ~20 requests/min; paid models need no pause
PAUSE_S = 15 if openrouter_model.endswith(":free") else 0

async def run_with_retry(agent, prompt, max_retries=5, wait_s=60):
    for attempt in range(max_retries):
        TRACE.clear()  # discard partial traces from a failed attempt
        t0 = time.perf_counter()
        try:
            res = await agent.run(prompt)
            return res, time.perf_counter() - t0  # latency excludes rate-limit waits
        except ModelHTTPError as e:
            if e.status_code != 429 or attempt == max_retries - 1:
                raise
            print(f"Rate limited, waiting {wait_s}s (attempt {attempt + 1}/{max_retries})")
            await asyncio.sleep(wait_s)

def first_index(seq, item):
    return seq.index(item) if item in seq else None

async def evaluate(agent_to_eval, cases, n_runs=3, label="planner", pause_s=PAUSE_S):
    rows = []
    for case in cases:
        for run in range(n_runs):
            res, latency = await run_with_retry(agent_to_eval, case["q"])
            called = [t["to"] for t in TRACE]
            called_set, expect = set(called), case["expect"]
            usage = res.usage() if callable(res.usage) else res.usage

            row = {
                "system": label, "type": case["type"], "run": run,
                "correct": all(a in res.output for a in case["answer"]),
                "tokens": usage.total_tokens, "requests": usage.requests,
                "latency_s": round(latency, 2),
            }
            if label == "planner":
                row |= {
                    "route_exact": called_set == expect,
                    "route_recall": len(called_set & expect) / len(expect) if expect else 1.0,
                    "route_precision": len(called_set & expect) / len(called_set) if called_set else float(not expect),
                    "redundant_calls": len(called) - len(called_set),
                    "inner_tool_used": all(
                        ("calculator" in t["inner_tools"]) if t["to"] == "teacher"
                        else ("search_course_docs_tool" in t["inner_tools"])
                        for t in TRACE),
                }
                if "order" in case:
                    i_docs, i_teach = (first_index(called, a) for a in case["order"])
                    row["order_ok"] = None not in (i_docs, i_teach) and i_docs < i_teach
                if "handoff" in case:
                    row["handoff_ok"] = any(case["handoff"] in t["query"]
                                            for t in TRACE if t["to"] == "teacher")
            rows.append(row)
            await asyncio.sleep(pause_s)
    return pd.DataFrame(rows)


In [ ]:
# Baseline: one agent with both tools and no delegation
single_agent = Agent(
    model=OpenAIChatModel(model_name=openrouter_model, provider=openrouter_provider),
    instructions="You answer calculation and course questions using your tools.",
)
single_agent.tool_plain(calculator)               # the teacher's tool function
single_agent.tool_plain(search_course_docs_tool)  # the coordinator's tool function

team = await evaluate(planning_agent, CASES, label="planner")
solo = await evaluate(single_agent, CASES, label="single")

print(team.groupby("type").mean(numeric_only=True).round(2))
summary = pd.concat([team, solo]).groupby("system")[["correct", "tokens", "latency_s"]].mean()
print(summary)
print("Coordination gain (accuracy):",
      summary.loc["planner", "correct"] - summary.loc["single", "correct"])
print("Token overhead:", summary.loc["planner", "tokens"] / summary.loc["single", "tokens"])